# 🧠 Trinity Hippocampal Learning Probe (THLP) - Benchmark Task

**DeepMind AGI Hackathon 2026**

This task evaluates hippocampal learning through error-driven belief updating across 5 cognitive domains:
- Causal Inference
- Belief Revision
- Counterfactual Reasoning
- Analogical Mapping
- Meta-Learning

**Scoring:**
- Accuracy (60%): Binary correct/incorrect per item
- ECE (20%): Expected Calibration Error
- Brier Score (20%): Mean squared error of probabilities

**Dataset:** 2,400 test items, φ-scaled difficulty (3, 5, 8, 13, 21)

**Expected Baselines:**
- Claude 3.5 Sonnet: ~64% accuracy
- Nemotron 120B: ~22% accuracy


In [ ]:
# Import libraries
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
from typing import Dict, Any

print("✅ THLP Benchmark Ready")

## Part 1: Load Dataset

In [ ]:
# Load THLP dataset from Kaggle input
# The dataset is: playra/trinity-cognitive-probes-thlp
df = pd.read_csv("/kaggle/input/trinity-cognitive-probes-thlp/thlp_learning.csv")

print(f"📊 Loaded {len(df)} THLP items")
print(f"\nDifficulty distribution (phi levels):")
print(df['difficulty'].value_counts().sort_index())
print(f"\nSample item:")
print(df.iloc[0].to_dict())

## Part 2: Single Item Task

In [ ]:
@kbench.task(name="thlp_single_item")
def thlp_single_item(
    llm,
    question: str,
    options: str,
    correct_answer: str,
    difficulty: int,
    track: str
) -> Dict[str, Any]:
    """Evaluate a single THLP item.
    
    Args:
        llm: LLM model for inference
        question: The probe question
        options: Multiple choice options (A, B, C, D)
        correct_answer: Ground truth answer
        difficulty: Phi-level difficulty (3, 5, 8, 13, 21)
        track: Cognitive track identifier
    
    Returns:
        Dict with is_correct, confidence, and model response
    """
    # 1. Construct prompt with question and options
    prompt = f"""You are answering cognitive assessment questions.

Question: {question}

Options:
{options}

Respond with ONLY the letter of your answer (A, B, C, or D).
"""
    
    # 2. Prompt the LLM
    response = llm.prompt(prompt).strip().upper()
    
    # 3. Extract answer (first letter if multiple characters)
    predicted_answer = response[0] if response and response[0] in 'ABCD' else 'A'
    
    # 4. Grade the response
    is_correct = predicted_answer == correct_answer.upper()
    
    # 5. Assert for pass/fail
    kbench.assertions.assert_true(
        is_correct,
        expectation=f"Model answer '{predicted_answer}' should match '{correct_answer}'"
    )
    
    return {
        "is_correct": is_correct,
        "predicted_answer": predicted_answer,
        "correct_answer": correct_answer,
        "difficulty": difficulty,
        "model_response": response
    }

# Test with a single item
# thlp_single_item.run(
#     llm=kbench.llm,
#     question=df.iloc[0]['question'],
#     options=df.iloc[0]['options'],
#     correct_answer=df.iloc[0]['correct_answer'],
#     difficulty=df.iloc[0]['difficulty'],
#     track='thlp'
# )

## Part 3: Batch Evaluation Task (Main Task)

In [ ]:
@kbench.task(name="thlp_benchmark")
def thlp_benchmark(llm, df, max_items: int = 100) -> float:
    """Run THLP benchmark on dataset items.
    
    Args:
        llm: LLM model to evaluate
        df: DataFrame with THLP items
        max_items: Maximum number of items to evaluate (for testing)
    
    Returns:
        Accuracy score (0.0 to 1.0)
    """
    # Limit items for evaluation (full dataset = 2400)
    eval_df = df.head(max_items)
    
    print(f"\n🎯 Evaluating {len(eval_df)} THLP items...")
    
    # Enable caching for development speed
    with kbench.client.enable_cache():
        # Run thlp_single_item for each row
        runs = thlp_single_item.evaluate(
            stop_condition=lambda r: len(r) == len(eval_df),
            max_attempts=1,  # Fail fast during testing
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=5,  # Parallel evaluation
        )
    
    # Convert to DataFrame for analysis
    results_df = runs.as_dataframe()
    
    # Calculate accuracy
    accuracy = float(results_df.result.str.get("is_correct").mean())
    
    # Calculate per-difficulty accuracy
    results_df['difficulty'] = results_df.result.apply(lambda x: x.get('difficulty', 0))
    per_difficulty = results_df.groupby('difficulty')['is_correct'].mean()
    
    print(f"\n📊 Results:")
    print(f"  Overall Accuracy: {accuracy:.2%}")
    print(f"  By Difficulty:")
    for diff, acc in per_difficulty.items():
        print(f"    φ={diff}: {acc:.2%}")
    
    return accuracy

# Run benchmark evaluation
# _ = thlp_benchmark.run(kbench.llm, df, max_items=10)

## Part 4: Select Primary Task for Submission

This cell specifies `thlp_benchmark` as the task to save when you click **"Save Task"** in the top-right corner.

In [ ]:
%choose thlp_benchmark

## 🚀 Next Steps

1. Click **"Save Task"** in the top-right corner
2. Add to Benchmark: Trinity Cognitive Probes - THLP Learning Track
3. Configure models: Claude 3.5 Sonnet, GPT-4o, Gemini
4. Publish for hackathon jury evaluation

**Dataset URL:** https://www.kaggle.com/datasets/playra/trinity-cognitive-probes-thlp

**Benchmark URL:** https://www.kaggle.com/benchmarks/playra-trinity-cognitive-probes-thlp